<a href="https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-Farooq292/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm choosing a Decision Tree (max_depth=3 or 4) as my model. This fits Lane 2 well for two reasons:
  (1) it directly extends the depth-2 tree I already explored in Notebook 02, so I can see whether more depth genuinely improves ranking without becoming unreadable.
  (2) decision trees produce an if/else structure a content reviewer could actually read and trust, unlike a black-box model — which matters for a decision-support tool where someone needs to understand *why* a page was flagged, not just trust a score blindly.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, subprocess

if not os.path.isdir("flyrank-ml-internship"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Abdullah-Farooq292/flyrank-ml-internship"], check=True)
os.chdir("flyrank-ml-internship")

import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")
print(df.columns.tolist())


Loaded 30000 rows
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'm using a client-grouped split rather than a plain random split. Even though my label isn't a future-window prediction (so a time-aware split isn't strictly necessary here), pages from the same client likely share similar patterns (same site structure, same content strategy). If pages from one client end up in both train and test, the model could partly "memorize" that client instead of learning signal that generalizes to new clients — which would make my Precision@K look better than it really is. Grouping by client_id keeps the test set honest.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]

X = df[features].replace([float("inf"), float("-inf")], None).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Unique clients in train: {df['client_id'].iloc[train_idx].nunique()}")
print(f"Unique clients in test: {df['client_id'].iloc[test_idx].nunique()}")

# Confirm no overlap between train and test clients
overlap = set(df['client_id'].iloc[train_idx]) & set(df['client_id'].iloc[test_idx])
print(f"Client overlap between train/test (should be 0): {len(overlap)}")


Train rows: 22885, Test rows: 7115
Unique clients in train: 24
Unique clients in test: 8
Client overlap between train/test (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

This trains a Decision Tree on the client-grouped split above, then compares its Precision@20 and Precision@50 against my Week-4 baseline rule (stale_and_visible_score), measured on the exact same test set for a fair comparison.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.tree import DecisionTreeClassifier, export_text

# Train the model
tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

# Get model scores on the test set
model_scores = tree.predict_proba(X_test)[:, 1]

# Rebuild baseline score on the same test rows (from Assignment 5's rule)
test_df = df.iloc[test_idx]
baseline_scores = (
    (test_df["days_since_last_update"] >= 180).astype(int)
    * (test_df["impressions_90d"] >= 500).astype(int)
    * test_df["impressions_90d"]
)

def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Model vs Baseline — Precision@K on held-out clients:")
for k in (20, 50):
    model_p = precision_at_k(model_scores, y_test.values, k)
    baseline_p = precision_at_k(baseline_scores.values, y_test.values, k)
    print(f"Precision@{k}: baseline {baseline_p:.3f}  vs  model {model_p:.3f}")

print()
print("Tree structure:")
print(export_text(tree, feature_names=features))


Model vs Baseline — Precision@K on held-out clients:
Precision@20: baseline 0.500  vs  model 0.600
Precision@50: baseline 0.620  vs  model 0.600

Tree structure:
|--- impressions_90d <= 7.50
|   |--- avg_position <= 1.65
|   |   |--- avg_position <= 0.25
|   |   |   |--- word_count <= 669.50
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  669.50
|   |   |   |   |--- class: 0
|   |   |--- avg_position >  0.25
|   |   |   |--- word_count <= 3134.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  3134.00
|   |   |   |   |--- class: 0
|   |--- avg_position >  1.65
|   |   |--- content_age_days <= 108.50
|   |   |   |--- days_since_last_update <= 4.50
|   |   |   |   |--- class: 1
|   |   |   |--- days_since_last_update >  4.50
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- impressions_90d <= 3.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  3.50
|   |   |   |   |--- class: 0
|--- impressions_90d >  7.50


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Comparing the two methods: my baseline rule wins narrowly at Precision@50 (0.620 vs 0.600), while the tree wins at Precision@20 (0.600 vs 0.500). This means my hand-written rule is actually quite strong within the top 50 — likely because "stale + visible" is a genuinely powerful signal on its own, as I confirmed in Assignment 5 (94% decline rate). The tree found a different pattern: it leans most heavily on impressions_90d as its first split, then avg_position, ctr, and content_age_days — mixing several signals in ways my single hand-written rule cannot. This suggests the tree isn't dramatically better here, but it captures a different, complementary slice of the "declining pages" pattern rather than a strictly superior one. Neither method is dominant across both metrics, which is a more honest and realistic outcome than a model sweeping every metric.

What the model leans on: the tree's very first split is impressions_90d <= 7.5 — pages with almost no traffic get bucketed together immediately, regardless of other features. Deeper in the tree, ctr and avg_position matter most for higher-traffic pages, while content_age_days and days_since_last_update matter more for low-traffic, low-position pages.

This is confirmed by direct inspection: there are 2,896 test pages where the model is confident (probability > 0.7) that a page is declining, but my baseline rule assigns it a score of 0 because it doesn't meet the "stale + visible" threshold. Of those 2,896 pages, 58.6% are genuinely declining — meaningfully above chance. This shows the model is catching a real pattern that my single hand-written rule structurally cannot see, since my rule only fires on one specific combination of signals (staleness + high traffic), while the tree can combine impressions, CTR, position, and content age in more flexible ways.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Look at a few cases where the model disagrees most with the baseline
comparison = pd.DataFrame({
    "actual_label": y_test.values,
    "model_score": model_scores,
    "baseline_score": baseline_scores.values
})

# Model confident it's declining, but baseline says no signal at all
model_only = comparison[(comparison["model_score"] > 0.7) & (comparison["baseline_score"] == 0)]
print(f"Cases where model is confident but baseline sees nothing: {len(model_only)}")
print(f"Of those, actually declining: {model_only['actual_label'].mean():.3f}")


Cases where model is confident but baseline sees nothing: 2896
Of those, actually declining: 0.586


## Self-check

Before you submit, confirm each line honestly:

- [done ] Every section above is filled — markdown thinking AND the code that backs it
- [done ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [done ] No client names, URLs, or private queries anywhere
- [done ] My claims use careful words: observed, measured, directional, decision-support
- [done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.